In [ ]:
#@title << Run this cell first — environment setup {display-mode: "form"}
import sys, os

if "google.colab" in sys.modules:
    !git clone --quiet --single-branch --branch main https://github.com/YanickSchraner/CAS-DeepRL.git
    !cp -r "CAS-DeepRL/Tag 2/envs" .
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

> 📝 **Homework / Challenge** — Complete this notebook after the Policy Gradient lecture (notebook `01_PolicyGradient`). It applies the REINFORCE algorithm you coded in class to a real building-control problem and sets up *why* we add a critic in the next notebook.

# Challenge: REINFORCE for Smart Building Control 🏢🌡️

## The Business Problem

You just coded **REINFORCE** (Monte-Carlo Policy Gradient) from scratch on CartPole — a toy balancing task. Now you'll apply the **exact same algorithm** to a real business problem: **HVAC control**.

**The setting:** You control the heater in a room. Every 15 minutes you choose a heater setting. You want to keep the room comfortable (around **21 °C**) while spending as little as possible on electricity — and electricity is *expensive in the early evening, cheap at night*. The two goals fight each other: heat is cheapest exactly when you need it least.

**Real-world analogs:**
- **Google DeepMind** used RL to cut data-centre cooling energy by **~40 %** (2016).
- **Smart thermostats** (Nest, Ecobee) and commercial **building-management systems** increasingly use learned controllers.
- Demand-response programs pay buildings to **pre-heat / pre-cool** around price peaks.

The traditional solution is a **thermostat** (bang-bang control: heat if too cold, stop if warm enough). It keeps you comfortable but ignores electricity prices entirely. Can a policy-gradient agent learn a smarter, price-aware strategy?

## Your Mission

You already know REINFORCE. The challenge here is:
1. Apply it to a new environment (continuous observations — no Q-table possible!).
2. Understand what state, action, and reward mean in **building-control** terms.
3. Beat the naive baselines (random, always-on, simple thermostat).
4. **Feel the pain of high variance** — and understand why the next notebook adds a *critic baseline* (A2C).

## Setup

In [ ]:
!pip install gymnasium torch numpy matplotlib tqdm --quiet

In [ ]:
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from envs.hvac_env import HvacEnv

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Part 1 — Understand the Environment

Before writing any learning code, understand the **five MDP components** in business terms — just like the inventory challenge on Day 1.

---

### The Environment

You manage the heater of a single room. Every **15 minutes** you pick a heater power level. A simple **RC thermal model** governs how the room warms up (from the heater) and cools down (heat leaking to the colder outdoors). The outdoor temperature follows a daily cycle, and the **electricity price** changes through the day — cheap overnight, peaking in the early evening. Small random fluctuations make the dynamics noisy.

---

### Observation Space — what the agent sees each step

| Signal | Range | Meaning |
|---|---|---|
| `room_temp` | [10, 35] °C | Current room temperature — the thing you control |
| `outdoor_temp` | [-10, 35] °C | Outside temperature — drives how fast heat leaks away |
| `hour_sin` | [-1, 1] | Sine-encoded hour of day |
| `hour_cos` | [-1, 1] | Cosine-encoded hour of day (paired with `hour_sin` to pinpoint the time without a midnight discontinuity) |
| `price_norm` | [0, 1] | Current electricity price, normalised (0 = cheapest, 1 = peak) |

These five signals are **continuous**. On Day 1 you discretised the inventory state into bins and used a Q-table. Here that would mean binning 5 continuous dimensions — a combinatorial explosion. This is precisely why we use a **neural-network policy** instead: it reads the raw continuous vector directly. That is the whole point of *deep* RL.

---

### Actions — what the agent decides each step

The agent chooses one of **4 discrete heater settings**:

| Action | Setting | Power |
|---|---|---|
| 0 | OFF | 0.0 kW |
| 1 | LOW | 0.5 kW |
| 2 | MEDIUM | 1.5 kW |
| 3 | HIGH | 3.0 kW |

More power warms the room faster but costs more — and costs *much* more during the evening price peak.

---

### Reward Signal — what we optimise

Each 15-minute step earns:

```
reward = −(discomfort_penalty + cost_weight × electricity_cost)
```

| Component | Formula | Intuition |
|---|---|---|
| Discomfort penalty | \|room_temp − 21 °C\| × 2.0 | The further from 21 °C, the unhappier the occupants |
| Electricity cost | heater_kW × price_€/kWh × 0.25 h | Energy used this step × its price |
| `cost_weight` | 10.0 (default) | Scales the cost into the same range as discomfort, so saving money is genuinely worth it |

The reward is always ≤ 0 — you are minimising a *cost*. Comfort still dominates the daily total, but with `cost_weight = 10` the price of electricity matters enough that a good agent **shifts heating into cheap hours** instead of ignoring price like a plain thermostat.

---

### Episode — how long does one run last?

One episode = **one day = 96 steps** (24 h × 4 steps/h). The room starts each episode at a random temperature (16–26 °C) with a random outdoor baseline. There is no early termination; the agent maximises total reward (i.e. minimises total cost + discomfort) over the full day.

Run the next cell to inspect the spaces and watch a **random** agent — your weakest baseline.

In [ ]:
env = HvacEnv(seed=42)

print("=== HVAC Environment ===")
print(f"Observation space : {env.observation_space}")
print(f"  [room_temp, outdoor_temp, hour_sin, hour_cos, price_norm]")
print(f"Action space      : {env.action_space}  (0=OFF, 1=LOW, 2=MED, 3=HIGH)")
print(f"Episode length    : {env.episode_length} steps (= 1 day in 15-min intervals)\n")

obs, _ = env.reset(seed=42)
print("Sample observation:", np.round(obs, 2))
print(f"  room_temp    : {obs[0]:.1f} °C")
print(f"  outdoor_temp : {obs[1]:.1f} °C")
print(f"  hour_sin/cos : {obs[2]:.2f}, {obs[3]:.2f}")
print(f"  price_norm   : {obs[4]:.2f}  (0=cheapest, 1=peak)")

# Watch a random agent for one day
obs, _ = env.reset(seed=42)
temps, powers, prices, hours, rewards = [], [], [], [], []
done, hour = False, 0.0
while not done:
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    done = term or trunc
    temps.append(info["room_temp"]); powers.append(info["heater_kw"])
    prices.append(info["price_eur_kwh"]); rewards.append(r)
    hour += 0.25; hours.append(hour)

print(f"\nRandom agent total reward: {sum(rewards):.1f}")
print(f"Average room temp        : {np.mean(temps):.1f} °C  (target 21 °C)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(hours, temps, color="tomato", label="Room temp")
ax1.axhline(21, color="green", linestyle="--", label="Target 21 °C")
ax1.fill_between(hours, 20, 22, alpha=0.12, color="green", label="Comfort zone")
ax1.set_ylabel("Temperature (°C)"); ax1.legend(); ax1.set_title("Random Agent — One Day")
ax2.bar(hours, powers, width=0.2, color="steelblue")
ax2.set_ylabel("Heater (kW)"); ax2.set_xlabel("Hour of day")
plt.tight_layout(); plt.show()

> ❓ **Before you train:** What strategy would *you* use? When would you heat hard — and when would you coast? Would you pre-heat the room before the evening price peak so you can switch the heater off while electricity is expensive? Write down your intuition; you'll compare it to what the agent discovers.

## Part 2 — Build the Baselines

Before training any RL agent, **build baselines**. If REINFORCE can't beat simple rules, something is wrong. In building control the classic rule is a **thermostat** (bang-bang control).

In [ ]:
def run_episode(policy_fn, seed):
    """Run one episode with a policy function policy_fn(obs, env) -> action."""
    env = HvacEnv(seed=seed)
    obs, _ = env.reset()
    total, done = 0.0, False
    while not done:
        action = policy_fn(obs, env)
        obs, r, term, trunc, _ = env.step(action)
        total += r
        done = term or trunc
    return total


def evaluate_policy_fn(policy_fn, n_episodes=20, seed0=0):
    scores = [run_episode(policy_fn, seed0 + i) for i in range(n_episodes)]
    return float(np.mean(scores)), float(np.std(scores))


def random_policy(obs, env):
    return env.action_space.sample()

def always_off(obs, env):
    return 0

def always_medium(obs, env):
    return 2

def thermostat(obs, env):
    """Bang-bang thermostat: heat hard when cold, idle when warm. Ignores price."""
    room_temp = obs[0]
    if room_temp < 20.5:
        return 3      # HIGH
    elif room_temp < 21.0:
        return 2      # MEDIUM
    elif room_temp < 21.5:
        return 1      # LOW
    return 0          # OFF


baselines = {
    "Random"        : evaluate_policy_fn(random_policy),
    "Always OFF"    : evaluate_policy_fn(always_off),
    "Always MEDIUM" : evaluate_policy_fn(always_medium),
    "Thermostat"    : evaluate_policy_fn(thermostat),
}

print("Baseline comparison (mean reward over a 1-day episode):")
print("-" * 48)
for name, (m, s) in baselines.items():
    print(f"  {name:<14}: {m:9.1f} ± {s:.0f}")

> ❓ **Before training:** Which baseline do you expect REINFORCE to beat first? Why does *Always OFF* score so differently from *Always MEDIUM*? Which baseline do you think will be hardest to beat?

## Part 3 — Implement REINFORCE on HvacEnv

This is the **same algorithm** you coded in `01_PolicyGradient`. Two differences from the CartPole version:

1. **Continuous observations** → the policy network reads a 5-dim float vector (no change to the architecture, just the input size).
2. **Gymnasium API** → `env.reset()` returns `(obs, info)` and `env.step()` returns `(obs, reward, terminated, truncated, info)` (five values, not four).

Recall the REINFORCE recipe:
- Run a full episode with the **current stochastic policy**, saving each action's `log_prob`.
- Compute the **reward-to-go** return $G_t$ at every step (discounted sum of *future* rewards).
- Push the policy to make actions that led to high returns more likely: minimise $-\sum_t \log \pi(a_t|s_t)\, G_t$.

⚠️ **Note there is no value function and no baseline here** — the raw return $G_t$ is the only learning signal. Remember that as you watch training; it is the source of the variance you will measure in Part 7.

First, the **policy network**. Fill in the `TODO`s (then check the collapsed solution).

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super().__init__()
        # TODO: two fully connected layers: s_size -> h_size -> a_size
        # self.fc1 = ...
        # self.fc2 = ...

    def forward(self, x):
        # TODO: fc1 -> ReLU -> fc2 -> softmax over actions (dim=1)
        pass

    def act(self, obs):
        """Sample an action from the policy and return (action, log_prob)."""
        state = torch.from_numpy(obs).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()                 # sample, don't argmax — we need exploration!
        return action.item(), m.log_prob(action)

In [ ]:
#@title Solution — Policy network (click ▶ to expand) {display-mode: "form"}
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, obs):
        state = torch.from_numpy(obs).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

Now the **training loop**. Fill in the three `TODO`s: the environment interaction, the reward-to-go computation, and nothing else changes from the CartPole version.

In [ ]:
def reinforce(policy, optimizer, env, n_training_episodes, max_t, gamma, print_every=200):
    scores_deque = deque(maxlen=100)
    scores = []
    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []
        obs, _ = env.reset()
        for t in range(max_t):
            action, log_prob = policy.act(obs)
            saved_log_probs.append(log_prob)
            # TODO: take an environment step (Gymnasium returns 5 values)
            # obs, reward, terminated, truncated, _ = ...
            rewards.append(reward)
            if terminated or truncated:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Reward-to-go returns, computed back-to-front in O(N)
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            # TODO: G_t = r_t + gamma * G_{t+1}   (append to the LEFT)
            returns.appendleft(  )  # <-- complete this

        # Standardise returns (reduces variance, stabilises training)
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f"Episode {i_episode}\tAverage Score (last 100): {np.mean(scores_deque):.1f}")
    return scores

In [ ]:
#@title Solution — REINFORCE training loop (click ▶ to expand) {display-mode: "form"}
def reinforce(policy, optimizer, env, n_training_episodes, max_t, gamma, print_every=200):
    scores_deque = deque(maxlen=100)
    scores = []
    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []
        obs, _ = env.reset()
        for t in range(max_t):
            action, log_prob = policy.act(obs)
            saved_log_probs.append(log_prob)
            obs, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            if terminated or truncated:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(f"Episode {i_episode}\tAverage Score (last 100): {np.mean(scores_deque):.1f}")
    return scores

## Part 4 — Train Your Agent

Define the hyperparameters and train. On CPU this takes a couple of minutes; a GPU runtime is faster but not required (the network is tiny).

In [ ]:
hparams = {
    "s_size": 5,            # 5 continuous observations
    "a_size": 4,            # 4 heater settings
    "h_size": 128,
    "n_training_episodes": 2000,
    "max_t": 200,           # > 96, so episodes always run to their natural end
    "gamma": 0.99,
    "lr": 5e-3,
}

torch.manual_seed(42)
np.random.seed(42)

train_env = HvacEnv(seed=42)
policy = Policy(hparams["s_size"], hparams["a_size"], hparams["h_size"]).to(device)
optimizer = optim.Adam(policy.parameters(), lr=hparams["lr"])

scores = reinforce(
    policy, optimizer, train_env,
    hparams["n_training_episodes"], hparams["max_t"], hparams["gamma"],
    print_every=200,
)
print("\nTraining complete.")

In [ ]:
# Training curve vs. baselines  (Day-1 style: raw + moving average + baselines)
window = 50
smoothed = np.convolve(scores, np.ones(window) / window, mode="valid")

# "Always OFF" lets the room freeze -> a huge penalty that would squash the
# y-axis, so we leave it out of the plot (it is still in the table above).
EXCLUDE_FROM_PLOT = {"Always OFF"}
BASELINE_STYLE = {
    "Random":        {"color": "grey",   "linestyle": ":"},
    "Always MEDIUM": {"color": "orange", "linestyle": "--"},
    "Thermostat":    {"color": "green",  "linestyle": "-."},
}

plt.figure(figsize=(12, 4))
plt.plot(scores, alpha=0.15, color="steelblue")
plt.plot(range(window - 1, len(scores)), smoothed,
         color="steelblue", linewidth=2, label=f"REINFORCE ({window}-ep moving avg)")

for name, (m, _) in baselines.items():
    if name in EXCLUDE_FROM_PLOT:
        continue
    style = BASELINE_STYLE.get(name, {"color": "grey", "linestyle": "--"})
    plt.axhline(m, alpha=0.85, label=name, **style)

plt.xlabel("Episode")
plt.ylabel("Total reward (1-day)")
plt.title("REINFORCE Training Curve — HVAC Control")
plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Final evaluation
def reinforce_policy(obs, env):
    action, _ = policy.act(obs)
    return action

rl_mean, rl_std = evaluate_policy_fn(reinforce_policy, n_episodes=20)

print("Final evaluation (20 episodes):")
print("-" * 48)
for name, (m, s) in baselines.items():
    print(f"  {name:<14}: {m:9.1f} ± {s:.0f}")
print(f"  {'REINFORCE':<14}: {rl_mean:9.1f} ± {rl_std:.0f}  <- trained agent")
best = max(baselines.values(), key=lambda x: x[0])[0]
print(f"\n  vs best baseline: {rl_mean - best:+.1f}")

## Part 5 — Analyse the Learned Policy

What did the agent actually learn? Watch it run one day and compare its heating pattern to the electricity price.

In [ ]:
env_vis = HvacEnv(seed=7)
obs, _ = env_vis.reset()
temps, powers, prices, hours = [], [], [], []
done, hour = False, 0.0
while not done:
    action, _ = policy.act(obs)
    obs, _, term, trunc, info = env_vis.step(action)
    done = term or trunc
    temps.append(info["room_temp"]); powers.append(info["heater_kw"])
    prices.append(info["price_eur_kwh"]); hour += 0.25; hours.append(hour)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(hours, temps, color="tomato")
axes[0].axhline(21, color="green", linestyle="--", label="Target 21 °C")
axes[0].fill_between(hours, 20, 22, alpha=0.15, color="green")
axes[0].set_ylabel("Temperature (°C)"); axes[0].legend()
axes[0].set_title("Trained REINFORCE Agent — One Day")
axes[1].bar(hours, powers, width=0.2, color="steelblue")
axes[1].set_ylabel("Heater (kW)")
axes[2].plot(hours, prices, color="orange")
axes[2].set_ylabel("Price (€/kWh)"); axes[2].set_xlabel("Hour of day")
plt.tight_layout(); plt.show()

print(f"Average room temp : {np.mean(temps):.1f} °C")
print(f"Total energy cost : €{sum(p*0.25*pr for p, pr in zip(powers, prices)):.2f}")

> ❓ **Discussion:**
> 1. Does the agent keep the room near 21 °C? How does it compare to the thermostat baseline?
> 2. Does it **pre-heat** before the evening price peak, then coast through the expensive hours? Or does it behave like a naive thermostat that ignores price?
> 3. Where does REINFORCE still look erratic or sub-optimal?

In [ ]:
#@title 💡 Sample answer — analysing the policy (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**1. Does it hold 21 °C?** A trained REINFORCE agent should keep the room roughly in the comfort band and clearly beat *Random* and *Always OFF*. It usually lands **close to the thermostat** on comfort but is noticeably noisier — its stochastic policy occasionally picks a sub-optimal heater setting, so the temperature wobbles more than the deterministic thermostat's.

**2. Does it pre-heat before the price peak?** This is the prize behaviour. The reward weights electricity cost by `cost_weight = 10`, so price-shifting is a **real, secondary incentive** (comfort still dominates the daily total). A well-trained agent should therefore cluster *some* of its heating in the cheap early-morning hours and ease off during the evening peak — but REINFORCE's high variance means it often only *partially* discovers this. Honest caveat: on cold days, holding 21 °C needs close to the maximum heater power, leaving little room to coast through the peak, so the shifting you see may be modest. Compare the heater panel against the price panel and describe what is actually there.

**3. Where is it sub-optimal?** Expect occasional unnecessary heater bursts, temperature drift after the random start, and run-to-run inconsistency. That noise is not a bug in your code — it is the **high variance of the REINFORCE gradient**, which you'll quantify in Part 7. Be honest in your write-up: describe what the plot actually shows, not what you hoped to see.
"""))

## Part 6 — Reward Shaping Challenge 🎯

The default reward weights discomfort at **×2.0** and adds the raw electricity cost:

```
reward = −(|room_temp − 21| × 2.0 + electricity_cost)
```

**Business scenario:** Facilities management says: *"Comfort complaints are killing us. I don't care about the energy bill — keep this room at 21 °C, full stop."*

**Task:** Create a `HvacEnv` subclass with a **comfort-first** reward (heavily up-weight discomfort), train a new REINFORCE agent on it, and check:
1. Does the comfort-first agent hold temperature tighter (smaller average |T − 21|)?
2. What does it cost in extra electricity?

**How to do it:** subclass `HvacEnv` and override `step`. The parent's `info` dict already hands you the two reward components — `info["discomfort"]` and `info["cost_eur"]` — so you can re-weight them exactly.

In [ ]:
from envs.hvac_env import HvacEnv

class HvacEnvComfortFirst(HvacEnv):
    """Comfort-first reward variant."""
    COMFORT_FACTOR = 5.0   # multiply the discomfort penalty

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        # TODO: rebuild the reward so discomfort dominates.
        # info["discomfort"] = |room_temp - 21| * 2.0   (the parent's penalty)
        # info["cost_eur"]   = electricity cost this step
        custom_reward = reward   # <-- replace this
        return obs, custom_reward, terminated, truncated, info


# --- Your code: train a REINFORCE agent on HvacEnvComfortFirst and compare ---

In [ ]:
#@title Example Solution — comfort-first reward + comparison (click ▶ to expand) {display-mode: "form"}
from envs.hvac_env import HvacEnv

class HvacEnvComfortFirst(HvacEnv):
    """Comfort-first: discomfort is penalised 5x harder; energy almost ignored."""
    COMFORT_FACTOR = 5.0

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        custom_reward = -(self.COMFORT_FACTOR * info["discomfort"] + info["cost_eur"])
        return obs, custom_reward, terminated, truncated, info


# Train a fresh REINFORCE agent on the comfort-first reward
torch.manual_seed(42); np.random.seed(42)
comfort_env = HvacEnvComfortFirst(seed=42)
comfort_policy = Policy(hparams["s_size"], hparams["a_size"], hparams["h_size"]).to(device)
comfort_opt = optim.Adam(comfort_policy.parameters(), lr=hparams["lr"])
_ = reinforce(comfort_policy, comfort_opt, comfort_env,
              hparams["n_training_episodes"], hparams["max_t"], hparams["gamma"],
              print_every=500)


def rollout_stats(p, seed=7):
    """Average |T-21| and total energy cost for policy p over one day."""
    env = HvacEnv(seed=seed)          # evaluate both on the ORIGINAL dynamics
    obs, _ = env.reset()
    dev, cost, done = [], 0.0, False
    while not done:
        a, _ = p.act(obs)
        obs, _, term, trunc, info = env.step(a)
        done = term or trunc
        dev.append(abs(info["room_temp"] - 21.0))
        cost += info["heater_kw"] * 0.25 * info["price_eur_kwh"]
    return np.mean(dev), cost


orig_dev, orig_cost = rollout_stats(policy)
comf_dev, comf_cost = rollout_stats(comfort_policy)
print(f"Default agent      : avg |T-21| = {orig_dev:.2f} °C   energy cost = €{orig_cost:.2f}")
print(f"Comfort-first agent: avg |T-21| = {comf_dev:.2f} °C   energy cost = €{comf_cost:.2f}")
print("\nThe comfort-first agent should hug 21 °C more tightly (smaller |T-21|),")
print("paying for it with extra electricity. That trade-off IS the reward design choice.")

> ❓ **Reflect:** Who should decide the comfort-vs-cost weighting in a real building — the ML engineer or the facilities manager? What would change if electricity were 10× more expensive?

In [ ]:
#@title 💡 Sample answer — reward design (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**Who decides the weighting?** The **facilities manager / business owner**, not the ML engineer. The weight between comfort and cost is a *business value judgement*, not a technical one — it encodes how the organisation trades off occupant satisfaction against the energy bill. The engineer's job is to (a) make that trade-off **explicit and tunable**, and (b) show the manager the resulting curve (comfort vs cost) so they can pick a point. Hiding the weight inside the reward and choosing it yourself is how RL projects quietly optimise the wrong thing.

**If electricity were 10× more expensive,** the `electricity_cost` term would dominate, so the optimal policy would shift toward **aggressive price-shifting**: pre-heat hard in the cheap night/morning hours, then coast through the evening peak on low power, tolerating a little discomfort. The default reward (`cost_weight = 10`) already rewards *some* of this — comfort still leads, but price matters — which is why the agent doesn't behave like a price-blind thermostat. Cranking the weight (or the real price) up further is what tips it into the full DeepMind-style "shift load to cheap hours" behaviour.
"""))

## Part 7 — Why Is REINFORCE So Noisy? 🔍 (the punchline)

REINFORCE updates the policy using the **full Monte-Carlo return** $G_t$ as the signal. $G_t$ is a sum of many noisy rewards, so it has **high variance** — the same policy can get very different returns from one episode to the next, which makes the gradient jumpy and training unstable.

**Experiment:** train REINFORCE **three times** with different random seeds and overlay the curves. If the algorithm were low-variance, the three runs would look almost identical. Watch how much they differ.

In [ ]:
def quick_train(seed, n_episodes=800):
    torch.manual_seed(seed); np.random.seed(seed)
    env = HvacEnv(seed=100 + seed)
    p = Policy(hparams["s_size"], hparams["a_size"], hparams["h_size"]).to(device)
    opt = optim.Adam(p.parameters(), lr=hparams["lr"])
    return reinforce(p, opt, env, n_episodes, hparams["max_t"], hparams["gamma"], print_every=10_000)


seeds = [0, 1, 2]
runs = {sd: quick_train(sd) for sd in seeds}

window = 50
plt.figure(figsize=(12, 4))
for sd, sc in runs.items():
    sm = np.convolve(sc, np.ones(window) / window, mode="valid")
    plt.plot(range(window - 1, len(sc)), sm, linewidth=2, alpha=0.9, label=f"seed={sd}")
plt.axhline(baselines["Thermostat"][0], linestyle="-.", color="green", alpha=0.85, label="Thermostat")
plt.axhline(baselines["Random"][0], linestyle=":", color="grey", alpha=0.85, label="Random")
plt.xlabel("Episode"); plt.ylabel("Total reward (smoothed)")
plt.title("Three REINFORCE runs, identical settings — only the seed differs")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

print("Notice how differently the three runs progress. That spread is the")
print("high variance of the REINFORCE gradient estimator.")

> ❓ **Discussion:**
> 1. How different are the three curves? Would you trust a single run to compare two hyperparameter settings?
> 2. REINFORCE has *no baseline*. The Advantage Actor-Critic (A2C) you'll see in the next notebook subtracts a learned **value estimate** $V(s)$ from the return. Why would subtracting a baseline reduce variance without biasing the gradient?

In [ ]:
#@title 💡 Sample answer — variance & the critic baseline (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**1. How different are the runs?** Usually *very* — different learning speeds, different plateaus, sometimes one seed stalls while another takes off. The lesson: **never trust a single REINFORCE run** to compare two settings. You need several seeds and you should report the mean *and* the spread. High variance is the defining weakness of vanilla policy gradient.

**2. Why does subtracting a baseline help?** REINFORCE scales each action's log-prob by the raw return $G_t$. If every return in an episode is, say, a large negative number (as here — costs are always negative), *every* action gets pushed the same direction; the useful signal is only the *relative* difference between actions, drowned in a large common offset.

Subtracting a baseline $b(s)$ — the **critic's** value estimate $V(s)$ — replaces $G_t$ with the **advantage** $A_t = G_t - V(s_t)$: "was this action better or worse than what we *expected* from this state?" That centres the signal around zero, slashing variance.

Crucially it stays **unbiased**: because $b(s)$ depends only on the state and not the action, its expected contribution to the gradient is zero ($\mathbb{E}_a[
abla \log \pi(a|s)] \cdot b(s) = 0$). So you get the *same* gradient in expectation with *much less noise*.

That single idea — add a value-function baseline — is the step from REINFORCE to **Actor-Critic (A2C)**, which is exactly the next notebook. You've now felt the problem it solves. 🚀
"""))

## Summary

You applied **REINFORCE** — the same algorithm from CartPole — to a real building-control problem, with continuous observations that rule out a Q-table.

| What you did | Takeaway |
|---|---|
| Applied REINFORCE to HvacEnv | Policy gradient works on any MDP — and needs a neural policy for continuous states |
| Built baselines first (incl. a thermostat) | Rule-based controllers are your real competition |
| Analysed the learned policy | Interpretability builds stakeholder trust |
| Re-weighted the reward (comfort-first) | Reward design encodes a *business* value judgement |
| Measured variance across seeds | REINFORCE is high-variance — the core weakness of vanilla PG |

**Key insight for the next notebook:** REINFORCE learns, but it's noisy and sample-hungry because it has no baseline. Adding a learned **value-function baseline** (a *critic*) gives the **Advantage Actor-Critic (A2C)** algorithm — `02_ActorCritic`, where you'll control this *same* HVAC system with far more stability. 🚀